# 16 Deep CV: YOLO / Tracking / RTMPose / Bat / Plate / Homography

Purpose: create raw CV artifacts required for mechanics-style features after `12_full_cv_preprocess.ipynb` has created real `clips_v1` rows with `clip_path` values.

Required inputs:

- `clips/{FULL_RUN_ID}/clips_v1.parquet`

Created outputs:

- `detections/{FULL_RUN_ID}/detections_v1.parquet`
- `tracks/{FULL_RUN_ID}/tracks_v1.parquet`
- `pose2d/{FULL_RUN_ID}/pose2d_v1.parquet`
- `objects/{FULL_RUN_ID}/bat_detection_v1.parquet`
- `objects/{FULL_RUN_ID}/bat_line_v1.parquet`
- `homography/{FULL_RUN_ID}/homography_v1.parquet`

Heavy model downloads are disabled by default. Turn them on only in a GPU runtime after a tiny pilot.


In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('FULL_RUN_ID =', FULL_RUN_ID)


## Dependency Gate

For a real run, set `INSTALL_ULTRALYTICS=True` and `ENABLE_YOLO=True`. When `install_mmpose_default=True` and `enable_rtmpose=True`, this notebook installs the OpenMMLab stack needed for RTMPose in Colab before inference.


In [ ]:
import importlib.util
import shutil
import subprocess

DEEP_CV_SETTINGS = stage_settings(RUN_PROFILE, 'deep_cv')
INSTALL_ULTRALYTICS = bool(DEEP_CV_SETTINGS.get('install_ultralytics_default', True))
POSE_BACKEND = str(DEEP_CV_SETTINGS.get('pose_backend', 'rtmpose')).lower()
INSTALL_MEDIAPIPE = bool(DEEP_CV_SETTINGS.get('install_mediapipe_default', POSE_BACKEND == 'mediapipe'))
INSTALL_MMPOSE = bool(DEEP_CV_SETTINGS.get('install_mmpose_default', POSE_BACKEND == 'rtmpose'))
INSTALL_MMDET = bool(DEEP_CV_SETTINGS.get('install_mmdet_default', INSTALL_MMPOSE and POSE_BACKEND == 'rtmpose'))

if INSTALL_ULTRALYTICS and importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'])

if POSE_BACKEND == 'mediapipe' and INSTALL_MEDIAPIPE and importlib.util.find_spec('mediapipe') is None:
    print('Installing MediaPipe pose backend. This avoids the OpenMMLab/MIM Python 3.12 compatibility trap in current Colab runtimes.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe>=0.10.14'])
elif POSE_BACKEND == 'mediapipe':
    print('mediapipe already available; skipping pose backend install.')

if POSE_BACKEND == 'rtmpose' and INSTALL_MMPOSE and importlib.util.find_spec('mmpose') is None:
    print('Installing OpenMMLab dependencies for RTMPose. This can take several minutes in Colab.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip', 'setuptools', 'wheel'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'openmim'])
    mim_exe = shutil.which('mim')
    if mim_exe is None:
        raise RuntimeError('openmim was installed, but the mim command is not on PATH. Restart the Colab runtime and rerun this notebook.')
    subprocess.check_call([mim_exe, 'install', 'mmengine'])
    subprocess.check_call([mim_exe, 'install', 'mmcv>=2.0.1'])
    subprocess.check_call([mim_exe, 'install', 'mmpose>=1.1.0'])
elif POSE_BACKEND == 'rtmpose' and INSTALL_MMPOSE:
    print('mmpose already available; skipping OpenMMLab install.')

if POSE_BACKEND == 'rtmpose' and INSTALL_MMDET and importlib.util.find_spec('mmdet') is None:
    mim_exe = shutil.which('mim')
    if mim_exe is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'openmim'])
        mim_exe = shutil.which('mim')
    if mim_exe is None:
        raise RuntimeError('openmim was installed, but the mim command is not on PATH. Restart the Colab runtime and rerun this notebook.')
    subprocess.check_call([mim_exe, 'install', 'mmdet>=3.1.0'])

print('pose_backend =', POSE_BACKEND)
print('ultralytics available =', importlib.util.find_spec('ultralytics') is not None)
print('mediapipe available =', importlib.util.find_spec('mediapipe') is not None)
print('mmpose available =', importlib.util.find_spec('mmpose') is not None)
print('mmdet available =', importlib.util.find_spec('mmdet') is not None)


## Input Check


In [ ]:
from sport_pipeline.artifact_check import check_artifacts
from sport_pipeline.runtime.device import summarize_runtime_device

print(summarize_runtime_device(prefer_gpu=True, require_gpu=False))
required = [f'clips/{FULL_RUN_ID}/clips_v1.parquet']
print(check_artifacts(BASE_DIR, required))


## Run Deep CV

Safe default writes valid empty artifact contracts. For a real pilot, use:

- `ENABLE_YOLO=True`
- `ALLOW_MODEL_DOWNLOAD=True`
- `MAX_CLIPS=2` first

Add `OBJECT_MODEL` only when you have a bat/plate detector weight or want to try YOLO class-name hints.


In [ ]:
import importlib
import sport_pipeline.pipeline.preprocess.deep_cv as deep_cv_module

# Reload so a Drive-updated module is picked up in an already-running Colab runtime.
deep_cv_module = importlib.reload(deep_cv_module)
run_deep_cv_artifacts = deep_cv_module.run_deep_cv_artifacts

DEEP_CV_SETTINGS = stage_settings(RUN_PROFILE, 'deep_cv')
ENABLE_YOLO = bool(DEEP_CV_SETTINGS.get('enable_yolo', True))
ENABLE_RTMPOSE = bool(DEEP_CV_SETTINGS.get('enable_rtmpose', False))
ALLOW_MODEL_DOWNLOAD = bool(DEEP_CV_SETTINGS.get('allow_model_download', True))
YOLO_MODEL = str(DEEP_CV_SETTINGS.get('yolo_model', 'yolo11m.pt'))
POSE_BACKEND = str(DEEP_CV_SETTINGS.get('pose_backend', 'rtmpose')).lower()
RTMPOSE_MODEL = str(DEEP_CV_SETTINGS.get('rtmpose_model', 'rtmpose-l_8xb32-270e_coco-wholebody-384x288'))
RTMPOSE_DET_MODEL = DEEP_CV_SETTINGS.get('rtmpose_det_model')
MEDIAPIPE_MODEL_ASSET_PATH = DEEP_CV_SETTINGS.get('mediapipe_model_asset_path')
MEDIAPIPE_MODEL_ASSET_URL = str(DEEP_CV_SETTINGS.get('mediapipe_model_asset_url', 'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task'))
MEDIAPIPE_MODEL_COMPLEXITY = int(DEEP_CV_SETTINGS.get('mediapipe_model_complexity', 1))
MEDIAPIPE_MIN_DETECTION_CONFIDENCE = float(DEEP_CV_SETTINGS.get('mediapipe_min_detection_confidence', 0.3))
MEDIAPIPE_MIN_TRACKING_CONFIDENCE = float(DEEP_CV_SETTINGS.get('mediapipe_min_tracking_confidence', 0.3))
OBJECT_MODEL = DEEP_CV_SETTINGS.get('object_model')
TRACKER = str(DEEP_CV_SETTINGS.get('tracker', 'bytetrack.yaml'))
MAX_CLIPS = DEEP_CV_SETTINGS.get('max_clips')
MAX_FRAMES_PER_CLIP = DEEP_CV_SETTINGS.get('max_frames_per_clip')
REQUIRE_NON_EMPTY_DETECTIONS = bool(DEEP_CV_SETTINGS.get('require_non_empty_detections', True))
RESUME_DEEP_CV = bool(DEEP_CV_SETTINGS.get('resume', True))
CHECKPOINT_EVERY_CLIPS = int(DEEP_CV_SETTINGS.get('checkpoint_every_clips', 1))
CACHE_INPUTS = bool(DEEP_CV_SETTINGS.get('cache_inputs', False))
CACHE_NUM_WORKERS = int(DEEP_CV_SETTINGS.get('cache_num_workers', 4))
CACHE_MIN_FREE_DISK_GB = float(DEEP_CV_SETTINGS.get('cache_min_free_disk_gb', 20))
CACHE_MAX_FILE_MB = DEEP_CV_SETTINGS.get('cache_max_file_mb')

print('deep_cv enable_yolo =', ENABLE_YOLO, 'enable_pose =', ENABLE_RTMPOSE, 'pose_backend =', POSE_BACKEND)
print('deep_cv yolo_model =', YOLO_MODEL, 'rtmpose_model =', RTMPOSE_MODEL, 'rtmpose_det_model =', RTMPOSE_DET_MODEL)
print('deep_cv mediapipe_model_asset_path =', MEDIAPIPE_MODEL_ASSET_PATH)
print('deep_cv cache_inputs =', CACHE_INPUTS, 'cache_num_workers =', CACHE_NUM_WORKERS)

outputs = run_deep_cv_artifacts(
    BASE_DIR,
    FULL_RUN_ID,
    enable_yolo=ENABLE_YOLO,
    enable_rtmpose=ENABLE_RTMPOSE,
    yolo_model=YOLO_MODEL,
    pose_backend=POSE_BACKEND,
    rtmpose_model=RTMPOSE_MODEL,
    rtmpose_det_model=RTMPOSE_DET_MODEL,
    mediapipe_model_asset_path=MEDIAPIPE_MODEL_ASSET_PATH,
    mediapipe_model_asset_url=MEDIAPIPE_MODEL_ASSET_URL,
    mediapipe_model_complexity=MEDIAPIPE_MODEL_COMPLEXITY,
    mediapipe_min_detection_confidence=MEDIAPIPE_MIN_DETECTION_CONFIDENCE,
    mediapipe_min_tracking_confidence=MEDIAPIPE_MIN_TRACKING_CONFIDENCE,
    object_model=OBJECT_MODEL,
    tracker=TRACKER,
    allow_model_download=ALLOW_MODEL_DOWNLOAD,
    max_clips=MAX_CLIPS,
    max_frames_per_clip=MAX_FRAMES_PER_CLIP,
    require_non_empty_detections=REQUIRE_NON_EMPTY_DETECTIONS,
    resume=RESUME_DEEP_CV,
    checkpoint_every_clips=CHECKPOINT_EVERY_CLIPS,
    cache_dir=CACHE_DIR,
    cache_inputs=CACHE_INPUTS,
    cache_num_workers=CACHE_NUM_WORKERS,
    cache_min_free_disk_gb=CACHE_MIN_FREE_DISK_GB,
    cache_max_file_mb=CACHE_MAX_FILE_MB,
)
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())


In [ ]:
expected = [
    f'detections/{FULL_RUN_ID}/detections_v1.parquet',
    f'tracks/{FULL_RUN_ID}/tracks_v1.parquet',
    f'pose2d/{FULL_RUN_ID}/pose2d_v1.parquet',
    f'objects/{FULL_RUN_ID}/bat_detection_v1.parquet',
    f'objects/{FULL_RUN_ID}/bat_line_v1.parquet',
    f'homography/{FULL_RUN_ID}/homography_v1.parquet',
    f'reports/preflight/deep_cv_{FULL_RUN_ID}_progress.json',
]
print(check_artifacts(BASE_DIR, expected))
print('NEXT: notebooks/17_deep_sequence_features.ipynb')
